# 웹 페이지 로더

웹 페이지의 HTML 콘텐츠를 크롤링하여 Document 객체로 변환하는 방법을 학습합니다.

`WebBaseLoader`는 웹 기반 문서를 로드하는 범용 로더로, BeautifulSoup을 사용하여 HTML을 파싱하고 텍스트를 추출합니다.

온라인 뉴스, 블로그, 기술 문서 등을 RAG 시스템에 활용할 수 있습니다.


# 07. 웹 페이지 로더

## 학습 목표
- `WebBaseLoader`로 URL에서 HTML 콘텐츠를 가져와 Document로 변환해요
- `SoupStrainer`로 필요한 HTML 요소만 선택적으로 파싱해요
- 여러 URL을 동시에 비동기로 로딩하는 방법을 익혀요

## 사전 지식
- 00-Document-Loader 노트북 완료
- `pip install beautifulsoup4 requests` 설치

---

> **RAG 파이프라인 위치**: Load 단계 — 파일이 아닌 **웹 URL**에서 직접 데이터를 가져와요.


In [1]:
from dotenv import load_dotenv
load_dotenv()

# 예제 URL (위키백과 - 기계 학습 문서)
TEST_URL = "https://en.wikipedia.org/wiki/Machine_learning"


## 1. WebBaseLoader 기본 사용

`WebBaseLoader`는 웹 페이지의 HTML을 다운로드하고 BeautifulSoup으로 파싱하여 텍스트를 추출합니다.

**필요한 패키지**: `pip install beautifulsoup4 requests`

**주요 특징**:
- HTML 자동 파싱
- 메타데이터 자동 추출 (title, language 등)
- 단일/복수 URL 지원


> 🔑 **핵심 개념**: `WebBaseLoader`는 내부적으로 BeautifulSoup으로 HTML을 파싱합니다. `SoupStrainer`를 사용하면 전체 페이지가 아니라 본문 영역만 선택적으로 파싱해서 노이즈(네비게이션, 광고 등)를 제거할 수 있습니다.

In [2]:
# ============================================================
# TODO: WebBaseLoader로 웹 페이지를 Document로 로드해보세요
# 힌트: WebBaseLoader(TEST_URL) → loader.load()
# 예상 결과: 1개의 Document가 생성되고 title 등 메타데이터가 포함됩니다
# ============================================================

from langchain_community.document_loaders import WebBaseLoader

# 1단계: WebBaseLoader 생성
# - URL을 전달하면 HTML을 자동으로 다운로드하고 BeautifulSoup으로 파싱
loader = WebBaseLoader(TEST_URL)

# 2단계: 웹 페이지 로드
docs = loader.load()

# 3단계: 결과 확인
print(f"로드된 문서 개수: {len(docs)}")
print(f"\n첫 번째 문서의 타입: {type(docs[0])}")
print("=" * 60)
print("📄 텍스트 길이 및 미리보기:")
print("=" * 60)
print(f"전체 텍스트 길이: {len(docs[0].page_content)} 글자")
print(f"\n내용 미리보기 (처음 300자):")
print(docs[0].page_content[:300])
print("\n" + "=" * 60)
print("🏷️  메타데이터:")
print("=" * 60)
print(docs[0].metadata)

USER_AGENT environment variable not set, consider setting it to identify your requests.


로드된 문서 개수: 1

첫 번째 문서의 타입: <class 'langchain_core.documents.base.Document'>
📄 텍스트 길이 및 미리보기:
전체 텍스트 길이: 124973 글자

내용 미리보기 (처음 300자):




Machine learning - Wikipedia



























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload file

🏷️  메타데이터:
{'source': 'https://en.wikipedia.org/wiki/Machine_learning', 'title': 'Machine learning - Wikipedia', 'language': 'en'}


## 2. BeautifulSoup으로 특정 요소만 추출

`bs4.SoupStrainer`를 사용하면 HTML의 특정 요소만 선택적으로 파싱할 수 있습니다.

이를 통해 불필요한 네비게이션, 광고, 푸터 등을 제외하고 핵심 콘텐츠만 추출할 수 있습니다.

**시나리오**: 기술 블로그에서 본문(`article` 태그)만 추출하기


In [3]:
# ============================================================
# TODO: SoupStrainer로 Wikipedia 본문 영역만 추출해보세요
# 힌트: bs_kwargs=dict(parse_only=bs4.SoupStrainer("div", attrs={"class": ["mw-parser-output"]}))
# 예상 결과: 전체 페이지보다 짧은 텍스트 길이가 출력됩니다
# ============================================================

import bs4

# 기술 블로그 예제 (실제 URL로 변경 가능)
BLOG_URL = "https://en.wikipedia.org/wiki/Artificial_intelligence"

# 1단계: SoupStrainer로 파싱할 요소 지정
# - bs_kwargs: BeautifulSoup에 전달할 추가 옵션
# - parse_only: 이 필터에 해당하는 요소만 파싱 (나머지 HTML은 무시)
loader = WebBaseLoader(
    web_paths=(BLOG_URL,),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class": ["mw-parser-output"]},  # Wikipedia 본문 영역
        )
    ),
)

# 2단계: 로드 및 확인
docs = loader.load()

print(f"문서 개수: {len(docs)}")
print(f"텍스트 길이: {len(docs[0].page_content)} 글자")
print(f"\n💡 SoupStrainer를 사용하면 필요한 부분만 추출하여 텍스트 길이를 줄일 수 있습니다.")

문서 개수: 1
텍스트 길이: 0 글자

💡 SoupStrainer를 사용하면 필요한 부분만 추출하여 텍스트 길이를 줄일 수 있습니다.


## 3. User-Agent 설정 및 SSL 우회

일부 웹사이트는 크롤러를 차단하거나 SSL 인증을 요구할 수 있습니다.

이 경우 `header_template`으로 User-Agent를 설정하거나, `requests_kwargs`로 SSL 인증을 우회할 수 있습니다.


In [4]:
# User-Agent를 설정하여 일반 브라우저처럼 위장
loader = WebBaseLoader(
    web_paths=(TEST_URL,),
    header_template={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/102.0.0.0 Safari/537.36",
    },
)

# SSL 인증 우회 (필요한 경우)
loader.requests_kwargs = {"verify": True}  # False로 설정하면 SSL 검증 우회

docs = loader.load()
print(f"✅ User-Agent 설정 및 SSL 인증 처리 완료")
print(f"문서 개수: {len(docs)}")


✅ User-Agent 설정 및 SSL 인증 처리 완료
문서 개수: 1


> ⚠️ **자주 하는 실수**: 웹 크롤링 시 `robots.txt`를 무시하거나 단기간에 많은 요청을 보내면 IP 차단될 수 있습니다. `requests_per_second=1` 이하로 설정하고 이용 약관을 반드시 확인하세요.

## 4. 여러 URL 동시에 로드

여러 웹 페이지를 한 번에 로드할 수 있습니다.

URL 리스트를 전달하면, 각 URL이 개별 Document로 변환됩니다.


> 💡 **실무 팁**: JavaScript로 렌더링되는 SPA(React/Vue 기반) 페이지는 `WebBaseLoader`로 가져오면 빈 내용이 나옵니다. 그런 경우에는 Playwright나 Selenium 기반 로더를 사용해야 합니다.

In [5]:
# ============================================================
# TODO: URL 리스트를 WebBaseLoader에 전달해서 여러 페이지를 한 번에 로드해보세요
# 힌트: WebBaseLoader(urls) → loader.load() → 각 URL이 별도 Document로 변환됨
# 예상 결과: 3개의 Document가 생성되고 각각의 source URL이 메타데이터에 포함됩니다
# ============================================================

# 1단계: 여러 URL을 리스트로 전달
urls = [
    "https://en.wikipedia.org/wiki/Deep_learning",
    "https://en.wikipedia.org/wiki/Natural_language_processing",
    "https://en.wikipedia.org/wiki/Computer_vision"
]

loader = WebBaseLoader(urls)
docs = loader.load()

print(f"로드된 페이지 수: {len(docs)}")
print("\n" + "=" * 60)
for i, doc in enumerate(docs):
    print(f"\n[페이지 {i+1}]")
    print(f"URL: {doc.metadata.get('source')}")
    print(f"제목: {doc.metadata.get('title', 'N/A')}")
    print(f"텍스트 길이: {len(doc.page_content)} 글자")

로드된 페이지 수: 3


[페이지 1]
URL: https://en.wikipedia.org/wiki/Deep_learning
제목: Deep learning - Wikipedia
텍스트 길이: 137671 글자

[페이지 2]
URL: https://en.wikipedia.org/wiki/Natural_language_processing
제목: Natural language processing - Wikipedia
텍스트 길이: 54945 글자

[페이지 3]
URL: https://en.wikipedia.org/wiki/Computer_vision
제목: Computer vision - Wikipedia
텍스트 길이: 61449 글자


## 5. 비동기 로딩 (성능 향상)

여러 URL을 동시에 스크래핑하면 처리 속도를 크게 향상시킬 수 있습니다.

`aload()` 메서드를 사용하면 비동기로 여러 페이지를 병렬 다운로드합니다.

**주의**: `requests_per_second`로 초당 요청 수를 제한하여 서버 부하를 조절해야 합니다.


> 🎯 **강의 포인트**: 여러 URL을 리스트로 전달하면 각 URL이 별도의 Document로 변환됩니다. `metadata["source"]`에 원본 URL이 저장되므로, RAG 응답 시 "이 정보는 어떤 페이지에서 왔는지" 출처 표시가 자동으로 가능합니다.

In [6]:
import asyncio
import nest_asyncio
from typing import List

from langchain_core.documents import Document

# Jupyter Notebook에서 asyncio 사용을 위해 필요
nest_asyncio.apply()

# 여러 URL 준비
urls = [
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://en.wikipedia.org/wiki/Deep_learning",
]

loader = WebBaseLoader(urls)
loader.requests_per_second = 2  # 초당 2개 요청으로 제한 (서버 부하 방지)

# LangChain 0.3.14+에서는 aload가 동기 메서드로 변경되어 alazy_load를 직접 사용해야 함
async def fetch_async_docs(target_loader: WebBaseLoader) -> List[Document]:
    docs_buffer: List[Document] = []
    async for doc in target_loader.alazy_load():
        docs_buffer.append(doc)
    return docs_buffer

# 비동기 로드
docs = await fetch_async_docs(loader)

print(f"✅ 비동기 로딩 완료")
print(f"로드된 페이지 수: {len(docs)}")
print(f"\n💡 비동기 로딩은 여러 URL을 동시에 처리하여 시간을 단축합니다.")


Fetching pages: 100%|##########| 2/2 [00:00<00:00,  3.92it/s]


✅ 비동기 로딩 완료
로드된 페이지 수: 2

💡 비동기 로딩은 여러 URL을 동시에 처리하여 시간을 단축합니다.


## 6. 프록시 설정

IP 차단을 우회하거나 특정 네트워크 환경에서 크롤링할 때 프록시를 사용할 수 있습니다.

`proxies` 파라미터로 HTTP/HTTPS 프록시 서버를 지정합니다.


## 핵심 정리 및 마무리

### WebBaseLoader 주요 파라미터

| 파라미터 | 설명 | 예시 |
|---------|------|------|
| `web_paths` | URL(s) | `"https://..."` 또는 리스트 |
| `bs_kwargs` | BeautifulSoup 옵션 | `dict(parse_only=SoupStrainer(...))` |
| `header_template` | HTTP 헤더 | `{"User-Agent": "..."}` |
| `requests_per_second` | 초당 요청 수 제한 | `2` |

### 실무 팁
> 웹사이트를 크롤링할 때는 반드시 `robots.txt`를 확인하고 이용 약관을 준수해야 해요. `requests_per_second`를 2 이하로 설정해서 서버에 부담을 주지 않는 것이 중요해요.

### 주의 사항
> JavaScript로 동적으로 렌더링되는 페이지(SPA 등)는 `WebBaseLoader`로 가져올 수 없어요. 그런 경우에는 Selenium이나 Playwright가 필요해요.

---

## 다음 예고

다음은 가장 단순하지만 한국어 처리에 주의가 필요한 텍스트 파일 로더를 배워볼게요.

- **08-TXT-Loader**: 인코딩 처리 핵심 (utf-8, cp949, euc-kr)
- **09-JSON-Loader**: 구조화된 데이터에서 특정 필드 추출
